# Faz 2 — Baseline Model

**Adım 1:** Label Encoding + Train/Test Split (%80/%20)  
**Adım 2:** Pipeline — VarianceThreshold → StandardScaler  
**Adım 3:** Random Forest — 5-Fold Stratified CV

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import joblib
import json
import time
from pathlib import Path

# ── Google Colab / Drive kurulumu ─────────────────────────────────────────
try:
    import google.colab  # type: ignore
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    # Drive'da klasör: MyDrive/nids/outputs/
    # Buraya yükle: cicids2017_clean.parquet
    OUTPUTS = Path('/content/drive/MyDrive/nids/outputs')
except ImportError:
    IN_COLAB = False
    OUTPUTS = Path('../outputs')

OUTPUTS.mkdir(parents=True, exist_ok=True)
print(f"OUTPUTS → {OUTPUTS}  ({'Colab + Drive' if IN_COLAB else 'Yerel'})")
# ──────────────────────────────────────────────────────────────────────────

RESULTS_DIR = OUTPUTS / "results"
MODELS_DIR  = OUTPUTS / "models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


Mounted at /content/drive
OUTPUTS → /content/drive/MyDrive/nids/outputs  (Colab + Drive)


## 1. Veriyi Yükle

In [3]:
df = pd.read_parquet(OUTPUTS / "cicids2017_clean.parquet")
print(f"Satır: {len(df):,}  |  Sütun: {df.shape[1]}")
print(f"Sınıflar: {df['Label'].nunique()}")
df['Label'].value_counts()

Satır: 2,520,798  |  Sütun: 72
Sınıflar: 15


,count
Label,
BENIGN,2095057
DoS Hulk,172846
DDoS,128014
PortScan,90694
DoS GoldenEye,10286
FTP-Patator,5931
DoS slowloris,5385
DoS Slowhttptest,5228
SSH-Patator,3219


## 2. Label Encoding

In [4]:
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['Label'])

# Sınıf → integer eşlemesini kaydet
label_map = {cls: int(idx) for idx, cls in enumerate(le.classes_)}
with open(RESULTS_DIR / "label_map.json", "w") as f:
    json.dump(label_map, f, indent=2, ensure_ascii=False)

# Encoder'ı diske kaydet (Faz 3+ için tekrar kullanılacak)
joblib.dump(le, MODELS_DIR / "label_encoder.joblib")

print("Label map:")
for cls, idx in label_map.items():
    print(f"  {idx:2d} → {cls}")

Label map:
   0 → BENIGN
   1 → Bot
   2 → DDoS
   3 → DoS GoldenEye
   4 → DoS Hulk
   5 → DoS Slowhttptest
   6 → DoS slowloris
   7 → FTP-Patator
   8 → Heartbleed
   9 → Infiltration
  10 → PortScan
  11 → SSH-Patator
  12 → Web Attack � Brute Force
  13 → Web Attack � Sql Injection
  14 → Web Attack � XSS


## 3. Train / Test Split  (%80 / %20 — Stratified)

In [5]:
feature_cols = [c for c in df.columns if c not in ("Label", "label_enc")]
X = df[feature_cols]
y = df["label_enc"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train: {len(X_train):,} satır  ({len(X_train)/len(df)*100:.1f}%)")
print(f"Test : {len(X_test):,} satır  ({len(X_test)/len(df)*100:.1f}%)")
print(f"\nFeature sayısı: {X_train.shape[1]}")

Train: 2,016,638 satır  (80.0%)
Test : 504,160 satır  (20.0%)

Feature sayısı: 71


In [6]:
# Stratifikasyonu doğrula: her sınıfın oranı korunmuş mu?
train_dist = y_train.value_counts(normalize=True).rename("train_%")
test_dist  = y_test.value_counts(normalize=True).rename("test_%")
dist_check = pd.concat([train_dist, test_dist], axis=1).sort_index()
dist_check.index = le.inverse_transform(dist_check.index)
dist_check = dist_check * 100
dist_check["fark"] = (dist_check["train_%"] - dist_check["test_%"]).abs()
print(dist_check.round(4).to_string())
print(f"\nMaks. sapma: {dist_check['fark'].max():.5f}%  (0'a yakın olmalı ✓)")

                            train_%   test_%    fark
BENIGN                      83.1109  83.1109  0.0001
Bot                          0.0773   0.0774  0.0001
DDoS                         5.0783   5.0783  0.0000
DoS GoldenEye                0.4081   0.4080  0.0000
DoS Hulk                     6.8568   6.8568  0.0001
DoS Slowhttptest             0.2074   0.2075  0.0001
DoS slowloris                0.2136   0.2136  0.0000
FTP-Patator                  0.2353   0.2352  0.0000
Heartbleed                   0.0004   0.0004  0.0000
Infiltration                 0.0014   0.0014  0.0000
PortScan                     3.5978   3.5979  0.0000
SSH-Patator                  0.1277   0.1277  0.0000
Web Attack � Brute Force     0.0583   0.0583  0.0000
Web Attack � Sql Injection   0.0008   0.0008  0.0000
Web Attack � XSS             0.0259   0.0258  0.0001

Maks. sapma: 0.00010%  (0'a yakın olmalı ✓)


## 4. Pipeline: VarianceThreshold → StandardScaler

In [7]:
# VarianceThreshold eşiği: EDA'da 4 adet çok düşük varyanslı sütun tespit edildi.
# threshold=0.01 → bu sütunlar çıkarılır, geri kalan her şey StandardScaler'a girer.
preprocessor = Pipeline([
    ("var_thresh", VarianceThreshold(threshold=0.01)),
    ("scaler",     StandardScaler()),
])

preprocessor.fit(X_train)

# Kaç feature kaldı?
mask        = preprocessor.named_steps["var_thresh"].get_support()
kept_cols   = X_train.columns[mask].tolist()
dropped_cols = X_train.columns[~mask].tolist()

print(f"Başlangıç feature sayısı : {X_train.shape[1]}")
print(f"VarianceThreshold sonrası: {len(kept_cols)}  (düşürülen: {len(dropped_cols)})")
if dropped_cols:
    print(f"  Çıkarılan sütunlar: {dropped_cols}")

Başlangıç feature sayısı : 71
VarianceThreshold sonrası: 67  (düşürülen: 4)
  Çıkarılan sütunlar: ['Fwd URG Flags', 'RST Flag Count', 'CWE Flag Count', 'ECE Flag Count']


In [8]:
# Preprocessor'ı dönüştür → sonraki adımlarda (CV, SMOTE) hazır matris kullanılacak
X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f"X_train_proc shape: {X_train_proc.shape}")
print(f"X_test_proc  shape: {X_test_proc.shape}")
print(f"\nScaler mean  (ilk 5): {preprocessor.named_steps['scaler'].mean_[:5].round(4)}")
print(f"Scaler scale (ilk 5): {preprocessor.named_steps['scaler'].scale_[:5].round(4)}")

X_train_proc shape: (2016638, 67)
X_test_proc  shape: (504160, 67)

Scaler mean  (ilk 5): [8.6848741e+03 1.6597886e+07 1.0576800e+01 1.1969300e+01 6.0516420e+02]
Scaler scale (ilk 5): [1.90071235e+04 3.52348609e+07 8.23642900e+02 1.09782510e+03
 6.44565270e+03]


In [9]:
# Preprocessor'ı kaydet (Faz 3+ aynı pipeline'ı kullanacak)
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

# Kalan feature isimlerini kaydet
with open(RESULTS_DIR / "kept_features.json", "w") as f:
    json.dump(kept_cols, f, indent=2)

print("preprocessor.joblib  →", MODELS_DIR / "preprocessor.joblib")
print("kept_features.json   →", RESULTS_DIR / "kept_features.json")

preprocessor.joblib  → /content/drive/MyDrive/nids/outputs/models/preprocessor.joblib
kept_features.json   → /content/drive/MyDrive/nids/outputs/results/kept_features.json


## 5. Random Forest — 5-Fold Stratified CV

In [17]:
from sklearn.metrics import classification_report

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

n_splits   = 5
cv         = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
sample_frac = 0.1

# ── %10 örnekleme ────────────────────────────────────────────────────────
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(y_train), size=int(len(y_train) * sample_frac), replace=False)
idx.sort()

X_cv_full = X_train_proc[idx]
y_cv_full = y_train.iloc[idx].reset_index(drop=True)

# ── Çok nadir sınıfları çıkar ────────────────────────────────────────────
# Kök neden: Heartbleed(~1), SQL Injection(~2), Infiltration(~3) örnekleri
# n_splits'ten az → bazı fold'larda hiç yok → model o sınıf için sütun
# üretmiyor → y_prob sütun sayısı ≠ y_true sınıf sayısı → NaN.
# Çözüm: bu sınıfları CV havuzundan çıkar, standard scorer doğrudan çalışır.
class_counts = y_cv_full.value_counts()
rare_classes = class_counts[class_counts < n_splits].index.tolist()
if rare_classes:
    rare_names = [le.classes_[c] for c in rare_classes]
    print(f"CV'den çıkarılan çok nadir sınıflar (< {n_splits} örnek): {rare_names}")
    keep = ~y_cv_full.isin(rare_classes)
    X_cv = X_cv_full[keep.values]
    y_cv = y_cv_full[keep].reset_index(drop=True)
else:
    X_cv, y_cv = X_cv_full, y_cv_full
# ─────────────────────────────────────────────────────────────────────────

print(f"\nCV havuzu: {len(y_cv):,} satır  |  Sınıf sayısı: {y_cv.nunique()}")

print(f"CV başlıyor... ({len(y_cv):,} satır × 67 feature × {n_splits} fold)")
t0 = time.time()

cv_results = cross_validate(
    rf, X_cv, y_cv,
    cv=cv,
    scoring=["accuracy", "f1_macro", "roc_auc_ovr_weighted"],
    return_train_score=True,
    n_jobs=1,
    verbose=1,
)

elapsed = time.time() - t0
print(f"\nToplam süre: {elapsed/60:.1f} dakika")

CV'den çıkarılan çok nadir sınıflar (< 5 örnek): ['Infiltration', 'Web Attack � Sql Injection', 'Heartbleed']

CV havuzu: 201,657 satır  |  Sınıf sayısı: 12
CV başlıyor... (201,657 satır × 67 feature × 5 fold)

Toplam süre: 3.8 dakika


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  3.8min finished


In [18]:
# Fold sonuçlarını DataFrame'e aktar
folds_df = pd.DataFrame({
    "fold":               range(1, 6),
    "train_accuracy":     cv_results["train_accuracy"],
    "val_accuracy":       cv_results["test_accuracy"],
    "train_f1_macro":     cv_results["train_f1_macro"],
    "val_f1_macro":       cv_results["test_f1_macro"],
    "train_roc_auc":      cv_results["train_roc_auc_ovr_weighted"],
    "val_roc_auc":        cv_results["test_roc_auc_ovr_weighted"],
    "fit_time_s":         cv_results["fit_time"],
    "score_time_s":       cv_results["score_time"],
})

folds_df.to_csv(RESULTS_DIR / "random_forest_cv_folds.csv", index=False)
print(folds_df.to_string(index=False))

print("\n--- Ortalama ± Std ---")
for metric in ["val_accuracy", "val_f1_macro", "val_roc_auc"]:
    vals = folds_df[metric]
    print(f"  {metric:20s}: {vals.mean():.4f} ± {vals.std():.4f}")

 fold  train_accuracy  val_accuracy  train_f1_macro  val_f1_macro  train_roc_auc  val_roc_auc  fit_time_s  score_time_s
    1        0.999436      0.998488        0.994027      0.894168       0.999993     0.999874   40.410955      0.956248
    2        0.999504      0.998289        0.993937      0.912674       0.999994     0.999853   43.134507      0.561632
    3        0.999399      0.998364        0.991677      0.896176       0.999993     0.999850   42.884610      0.543947
    4        0.999455      0.998264        0.993209      0.906258       0.999994     0.999919   43.800379      0.610436
    5        0.999510      0.998140        0.992842      0.936421       0.999993     0.999848   43.117129      0.566064

--- Ortalama ± Std ---
  val_accuracy        : 0.9983 ± 0.0001
  val_f1_macro        : 0.9091 ± 0.0170
  val_roc_auc         : 0.9999 ± 0.0000


## 6. Fold Başına Classification Report — Hangi Sınıflar Düşük?

In [20]:
# Manuel CV döngüsü → fold başına classification_report
# CV havuzunda nadir sınıflar çıkarıldığı için y_cv.nunique() = 12 (≠ 15)
# labels + target_names parametreleriyle eşleştirme zorunlu.

cv_classes      = np.sort(y_cv.unique())           # CV'de var olan sınıf indisleri
cv_class_names  = le.classes_[cv_classes]          # karşılık gelen isimler

all_fold_reports = []

for fold_i, (train_idx, val_idx) in enumerate(cv.split(X_cv, y_cv), start=1):
    X_tr, X_val = X_cv[train_idx], X_cv[val_idx]
    y_tr, y_val = y_cv.iloc[train_idx], y_cv.iloc[val_idx]

    rf_fold = RandomForestClassifier(
        n_estimators=100, max_depth=None, min_samples_leaf=2,
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE,
    )
    rf_fold.fit(X_tr, y_tr)
    y_pred = rf_fold.predict(X_val)

    report = classification_report(
        y_val, y_pred,
        labels=cv_classes,           # sadece mevcut sınıflar
        target_names=cv_class_names, # eşleştirilmiş isimler
        output_dict=True,
        zero_division=0,
    )

    for cls_name in cv_class_names:
        all_fold_reports.append({
            "fold":      fold_i,
            "class":     cls_name,
            "precision": report[cls_name]["precision"],
            "recall":    report[cls_name]["recall"],
            "f1":        report[cls_name]["f1-score"],
            "support":   report[cls_name]["support"],
        })

    print(f"Fold {fold_i} macro-F1: {report['macro avg']['f1-score']:.4f}")

report_df = pd.DataFrame(all_fold_reports)

# Sınıf başına ortalama F1 — en düşükten yükseğe
mean_f1 = (
    report_df.groupby("class")
    .agg(mean_f1=("f1", "mean"), mean_support=("support", "mean"))
    .sort_values("mean_f1")
)
print("\n─── Sınıf Başına Ortalama F1 (düşükten yükseğe) ───")
print(mean_f1.to_string())

report_df.to_csv(RESULTS_DIR / "random_forest_cv_class_report.csv", index=False)

Fold 1 macro-F1: 0.8942
Fold 2 macro-F1: 0.9127
Fold 3 macro-F1: 0.8962
Fold 4 macro-F1: 0.9063
Fold 5 macro-F1: 0.9364

─── Sınıf Başına Ortalama F1 (düşükten yükseğe) ───
                           mean_f1  mean_support
class                                           
Web Attack � XSS          0.439183           9.2
Web Attack � Brute Force  0.759586          20.2
Bot                       0.777987          31.6
SSH-Patator               0.983019          49.0
DoS Slowhttptest          0.984046          95.2
DoS slowloris             0.986191          88.2
DoS GoldenEye             0.990176         162.8
PortScan                  0.994322        1472.6
DoS Hulk                  0.997027        2793.2
DDoS                      0.998979        2058.6
BENIGN                    0.999157       33461.0
FTP-Patator               1.000000          89.8
